# P60 — Por qué la mayoría de los hallazgos publicados son falsos

## 1. Título y paper

**Paper:** *Why Most Published Research Findings Are False*  
**Autoría:** John P. A. Ioannidis  
**Año y venue:** 2005 · PLoS Medicine, 2(8), e124  
**Nivel:** L3 · **Motor:** `valor_predictivo`  
**Ficha completa:** [`P60_valor_predictivo`](../../papers/foundational/P60_valor_predictivo/README.md)

**Hito:** Muestra con un modelo explícito que la probabilidad de que un hallazgo publicado sea cierto depende del diseño y de los incentivos, no del valor p.

- [DOI (PLoS Medicine)](https://doi.org/10.1371/journal.pmed.0020124)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La significancia estadística se leía como sinónimo de verdad. Nadie ponía número a la pregunta que de verdad importa: dado que se publicó, ¿qué probabilidad hay de que sea cierto?
2. Ejecutar una implementación mínima de la propuesta: Modelar el valor predictivo positivo en función de las odds previas, el poder estadístico, el nivel de significancia, el sesgo y el número de equipos que compiten.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Neyman y Pearson (1933), contraste de hipótesis
- Sterling (1959), sesgo de publicación


## 4. Intuición

Un resultado con p < 0,05 no tiene un 95 % de probabilidades de ser cierto. Esa probabilidad depende de cuántas hipótesis falsas se estaban probando, de cuánto poder tenía el estudio y de cuánta gente estaba compitiendo por publicarlo primero. Ioannidis le pone fórmula.


## 5. Concepto mínimo

```text
R   = odds previas de que la hipótesis sea cierta
1−β = poder estadístico            α = nivel de significancia

        PPV = (1−β)·R / (R − β·R + α)

Con sesgo u:   PPV = [(1−β)R + u·β·R] / [R + α − β·R + u − u·α + u·β·R]
Con n equipos: PPV = (R − R·βⁿ) / (R + 1 − (1−α)ⁿ − R·βⁿ)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('valor_predictivo', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué PPV tiene un exploratorio con poder 0,5 y odds previas 1:10?
2. ¿Qué le pasa si añadimos un 30 % de sesgo?
3. ¿Y si hay cinco equipos compitiendo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('valor_predictivo', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('valor_predictivo', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El exploratorio típico da PPV = 0,5: la mitad de esos hallazgos son falsos **antes** de contar el sesgo. Con un 30 % de sesgo cae a 0,1625, y con cinco equipos en carrera, a 0,2998. En un barrido masivo sin corrección el PPV es 0,0044.


## 10. Comentario pedagógico

Trasladado a la IA, las variables cambian de nombre pero no de papel: las odds previas son cuán plausible era la mejora, el poder es cuántas semillas y cuántos conjuntos se probaron, y el sesgo es la libertad para elegir la comparación favorable. [P63](../../papers/foundational/P63_reproducibilidad/README.md) ataca justamente ese sesgo.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer el valor p como «probabilidad de que la hipótesis sea falsa».


In [ ]:
print('p = P(dato tan extremo | hipotesis nula cierta)')
print('lo que se quiere es P(hipotesis cierta | dato)')
print('Para pasar de uno a otro hacen falta las odds previas. El valor p solo NO basta.')

## 12. Corrección

La cuenta correcta, con los escenarios del motor:


In [ ]:
r = run_paper_lab('valor_predictivo', seed=7)['result']
for e in r['escenarios']:
    print(f"{e['caso']:<38} PPV = {e['ppv']:<7} falso = {e['probabilidad_de_ser_falso']}")
print('mismo alfa = 0,05 en todas las filas.')

## 13. Desafío guiado

Localiza en la tabla `efecto_del_poder_con_R_0_1` cuánto sube el PPV al pasar el poder de 0,2 a 0,95, y explica por qué el poder importa más que el umbral de significancia.


In [ ]:
r = run_paper_lab('valor_predictivo', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica el modelo a un anuncio reciente de mejora en un benchmark de IA: estima R, poder, sesgo y número de equipos, calcula el PPV y documenta cada supuesto. El ejercicio no es acertar el número: es hacer explícitos los supuestos.


## 15. Evidencia de aprendizaje

Guarda la tabla de escenarios con su PPV y tu explicación de la diferencia entre α y P(hipótesis | dato).

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P60_valor_predictivo/README.md) · evaluación formal: [`assessments/papers/P60_valor_predictivo.md`](../../assessments/papers/P60_valor_predictivo.md)


## 16. Cierre

Ya hay un modelo de por qué se publican cosas falsas. Falta el otro filo del mismo problema: qué pasa cuando lo que se publica es cierto pero el corpus con el que se entrenó no representa a quien lo va a usar.


## 17. Conexión con el siguiente hito

- P63
- P62

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
